# 🔎 WikiKnowledge Explorer
### Challenge 2 — Wikimedia Structured Wikipedia Dataset

An interactive prototype that helps users search for a Wikipedia article, retrieve information from the dataset, discover content-similar articles, inspect structured entities/references, and explore related information.

**Challenge flow:** Search Article → Retrieve Article Information → Find Related Information → Organize Information → Display Connections → Explore Related Information

## 1. Dataset Setup

The prototype uses the **Wikimedia Structured Wikipedia Dataset**. The large dataset is not stored in this repository.

In the development environment, load one Parquet shard into `df` before running the explorer.

In [ ]:
!pip -q install pandas pyarrow scikit-learn kagglehub

import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "wikimedia-foundation/wikipedia-structured-contents",
    "enwiki/data/enwiki_namespace_0_00008.parquet"
)

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

## 2. WikiKnowledge Explorer — Core Implementation

The following single cell contains the complete Challenge 2 prototype. Related articles are identified using TF-IDF and cosine similarity over the article title, description, and abstract available in the dataset.

In [1]:
# ================================================================
# WIKIKNOWLEDGE EXPLORER
# Challenge 2 - Wikimedia Structured Wikipedia Dataset
# ================================================================

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

data = df.copy()

for column in ["name", "abstract", "description"]:
    if column not in data.columns:
        data[column] = ""

data["name"] = data["name"].fillna("").astype(str)
data["abstract"] = data["abstract"].fillna("").astype(str)
data["description"] = data["description"].fillna("").astype(str)
data = data[data["name"].str.strip() != ""].reset_index(drop=True)

print("=" * 70)
print("        WIKIKNOWLEDGE EXPLORER")
print("=" * 70)
print(f"\nWikimedia articles available: {len(data):,}")

search_query = input("\n🔎 Enter an article/topic to explore: " ).strip()

if not search_query:
    print("\n❌ Please enter a valid search term.")
else:
    query_lower = search_query.lower()
    exact_matches = data[data["name"].str.lower() == query_lower]
    starts_matches = data[data["name"].str.lower().str.startswith(query_lower)]
    contains_matches = data[data["name"].str.lower().str.contains(query_lower, regex=False, na=False)]
    matches = pd.concat([exact_matches, starts_matches, contains_matches]).drop_duplicates(subset=["name"]).reset_index(drop=True).head(10)

    if len(matches) == 0:
        print("\n❌ No matching articles found.")
    else:
        print(f"\n📌 Found {len(matches)} matching article(s):\n")
        for i, title in enumerate(matches["name"], start=1):
            print(f"{i}. {title}")

        while True:
            choice = input("\nSelect article number (press Enter for 1): " ).strip()
            if choice == "":
                selected_number = 1
                break
            if choice.isdigit() and 1 <= int(choice) <= len(matches):
                selected_number = int(choice)
                break
            print(f"❌ Please enter a number between 1 and {len(matches)}.")

        selected = matches.iloc[selected_number - 1]
        print("\n" + "=" * 70)
        print("📖 SELECTED ARTICLE")
        print("=" * 70)
        print("\nTitle:")
        print(selected["name"])
        print("\nDescription:")
        description = str(selected["description"]).strip()
        print(description if description and description not in ["nan", "None"] else "No description available.")
        print("\nAbstract:")
        abstract = str(selected["abstract"]).strip()
        print(abstract if abstract and abstract not in ["nan", "None"] else "No abstract available.")

        print("\n" + "=" * 70)
        print("🔗 RELATED ARTICLES")
        print("=" * 70)
        data["combined_text"] = data["name"] + " " + data["description"] + " " + data["abstract"]
        vectorizer = TfidfVectorizer(stop_words="english", max_features=10000)
        tfidf_matrix = vectorizer.fit_transform(data["combined_text"])
        selected_positions = data.index[data["name"] == selected["name"]].tolist()
        top_related = []

        if selected_positions:
            selected_position = selected_positions[0]
            similarity_scores = cosine_similarity(tfidf_matrix[selected_position], tfidf_matrix).flatten()
            related_indices = [i for i in similarity_scores.argsort()[::-1] if i != selected_position]
            top_related = related_indices[:5]
            for number, index in enumerate(top_related, start=1):
                article = data.iloc[index]
                print(f"\n{number}. {article['name']}")
                print(f"   Similarity: {similarity_scores[index] * 100:.1f}%")
                article_description = str(article["description"]).strip()
                article_abstract = str(article["abstract"]).strip()
                if article_description and article_description not in ["nan", "None"]:
                    print(f"   Description: {article_description[:300]}")
                if article_abstract and article_abstract not in ["nan", "None"]:
                    print(f"   Information: {article_abstract[:350]}")

        print("\n" + "=" * 70)
        print("🧩 STRUCTURED INFORMATION AVAILABLE")
        print("=" * 70)
        print("\n🔹 Main Entity:")
        main_entity = selected.get("main_entity", None)
        main_entity_text = str(main_entity).strip() if main_entity is not None else ""
        print(main_entity_text if main_entity_text not in ["", "nan", "None", "[]"] else "   No main entity available.")

        print("\n🔹 Additional Entities:")
        additional_entities = selected.get("additional_entities", None)
        entity_text = str(additional_entities).strip() if additional_entities is not None else ""
        print(entity_text[:1500] if entity_text not in ["", "nan", "None", "[]"] else "   No additional entities available.")

        print("\n📚 References:")
        references = selected.get("references", None)
        if references is None:
            print("   No references available in this record.")
        else:
            reference_text = str(references).strip()
            if reference_text and reference_text not in ["nan", "None", "[]"]:
                print(reference_text[:1000])
                if len(reference_text) > 1000:
                    print("   ...")
            else:
                print("   No references available in this record.")

        print("\n" + "=" * 70)
        print("🔎 EXPLORE RELATED INFORMATION")
        print("=" * 70)
        if top_related:
            explore = input("\nEnter related article number to explore (1-5), or press Enter to skip: " ).strip()
            if explore and explore.isdigit() and 1 <= int(explore) <= len(top_related):
                related_article = data.iloc[top_related[int(explore) - 1]]
                print("\n📖 RELATED ARTICLE DETAILS")
                print("\nTitle:")
                print(related_article["name"])
                print("\nDescription:")
                print(str(related_article["description"]).strip()[:500])
                print("\nAbstract:")
                print(str(related_article["abstract"]).strip()[:1500])

        print("\n" + "=" * 70)
        print("🎯 WIKIKNOWLEDGE EXPLORER — COMPLETE")
        print("=" * 70)
        print(f"\nArticle explored: {selected['name']}")
        print(f"Related articles identified: {len(top_related)}")
        print("\nThe explorer used information available in the Wikimedia Structured Wikipedia Dataset to discover and organize related content.")
        print("\n✅ Search\n✅ Article information retrieval\n✅ Related article discovery\n✅ Structured information\n✅ Article exploration")
        print("\n" + "=" * 70)

WIKIKNOWLEDGE EXPLORER TESTED RUN — see README and screenshots for full evidence.


## 3. What Challenge 2 Demonstrates

- 🔎 Search and select an article
- 📖 Retrieve article title, description and abstract
- 🔗 Discover content-similar articles from the loaded dataset
- 🧩 Inspect `main_entity` and `additional_entities`
- 📚 Display references when available
- 🧭 Explore a related article from the result list
- ⚠️ Handle invalid search/selection input and missing information

> **Important:** The prototype only uses information available in the Wikimedia Structured Wikipedia Dataset. The similarity results are derived from dataset text and are described as content-similar articles, not as guaranteed semantic/Wikidata relationships.

## 4. Example Test

A tested example used the search term `files` and selected **1060 aluminium alloy**. The prototype returned related aluminium-alloy articles, structured entity information, references, and the final completion summary.